In [1]:
# Complete Single-Neuron Implementation

In [2]:
import torch

# Data
X = torch.tensor([1., 2., 3., 4., 5.])
y = torch.tensor([3., 5., 7., 9., 11.])

# our dataset has relationship of y = 2x + 1

# Parameters
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

# Hyperparameter
learning_rate = 0.01

# Training
for epoch in range(1000):

    # Forward pass
    y_pred = w * X + b

    # Loss
    loss = torch.mean((y_pred - y) ** 2)

    # Clear old gradients
    if w.grad is not None:
        w.grad.zero_()
    if b.grad is not None:
        b.grad.zero_()

    # Backward pass
    loss.backward()

    # Update parameters
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    if epoch % 100 == 0:
        print(
            f"Epoch {epoch}, "
            f"Loss: {loss.item():.4f}, "
            f"w: {w.item():.4f}, "
            f"b: {b.item():.4f}"
        )

Epoch 0, Loss: 57.0000, w: 0.5000, b: 0.1400
Epoch 100, Loss: 0.0159, w: 2.0815, b: 0.7058
Epoch 200, Loss: 0.0081, w: 2.0581, b: 0.7903
Epoch 300, Loss: 0.0041, w: 2.0414, b: 0.8505
Epoch 400, Loss: 0.0021, w: 2.0295, b: 0.8935
Epoch 500, Loss: 0.0011, w: 2.0210, b: 0.9241
Epoch 600, Loss: 0.0005, w: 2.0150, b: 0.9459
Epoch 700, Loss: 0.0003, w: 2.0107, b: 0.9614
Epoch 800, Loss: 0.0001, w: 2.0076, b: 0.9725
Epoch 900, Loss: 0.0001, w: 2.0054, b: 0.9804


In [3]:
# instead of manual update lets use optimizer

In [5]:
optimizer = torch.optim.SGD(
    [w, b],
    lr=0.01
)

In [6]:
for epoch in range(1000):

    # Forward
    y_pred = w * X + b

    # Loss
    loss = torch.mean((y_pred - y) ** 2)

    # Clear gradients
    optimizer.zero_grad()

    # Backpropagation
    loss.backward()

    # Update parameters
    optimizer.step()

In [8]:
print('Weights: ',w.item())
print('Bias   : ',b.item())

Weights:  2.000131845474243
Bias   :  0.9995239973068237


In [9]:
x_new = torch.tensor(10.)

prediction = w * x_new + b

print(prediction)

tensor(21.0008, grad_fn=<AddBackward0>)


# 📚 Module 15.1 — Single Neuron — Notes

## 1. What is a Single Neuron? 🔥🔥🔥

A single neuron is the **smallest trainable building block** of a neural network.

It performs a weighted sum:

$$
z=wx+b
$$

For multiple inputs:

$$
z=\mathbf{w}^T\mathbf{x}+b
$$

Then an activation function may be applied:

$$
\hat{y}=f(z)
$$

For our first implementation, we use a **linear neuron**:

$$
\hat{y}=wx+b
$$

### Architecture

```text
        x
        │
        × w
        │
        ▼
      wx + b
        │
        ▼
       ŷ
        │
        ▼
      Loss
        │
        ▼
   Backpropagation
        │
        ▼
   Gradients
        │
        ▼
   Optimizer
        │
        ▼
    w and b
    updated
```

---

# 2. Parameters

The neuron has two learnable parameters:

```text
w → weight
b → bias
```

The model is:

$$
\hat y=wx+b
$$

During training, PyTorch changes $w$ and $b$ to reduce the loss.

---

# 3. Example Problem

Suppose the real relationship is:

$$
y=2x+1
$$

Training data:

| $x$ | $y$ |
| --: | --: |
|   1 |   3 |
|   2 |   5 |
|   3 |   7 |
|   4 |   9 |
|   5 |  11 |

The neuron initially **doesn't know** that:

$$
w=2,\qquad b=1
$$

It must learn these values from the examples.

---

# 4. Creating Data in PyTorch

```python
import torch

X = torch.tensor([1., 2., 3., 4., 5.])
y = torch.tensor([3., 5., 7., 9., 11.])
```

`X` contains inputs.

`y` contains target values.

Both are floating-point tensors because we're performing numerical optimization.

---

# 5. Creating Learnable Parameters 🔥🔥🔥

```python
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
```

### `requires_grad=True`

This tells PyTorch:

> Track operations involving this tensor so that gradients can later be calculated.

We need:

$$
\frac{\partial L}{\partial w}
$$

and:

$$
\frac{\partial L}{\partial b}
$$

---

# 6. Forward Pass 🔥🔥🔥

The neuron calculates:

```python
y_pred = w * X + b
```

Mathematically:

$$
\hat y=wx+b
$$

Initially:

$$
w=0,\quad b=0
$$

Therefore:

$$
\hat y=0
$$

for every input.

The model's predictions are initially far from the targets.

---

# 7. Loss Function

For this regression problem, we use **Mean Squared Error (MSE)**:

$$
L=\frac{1}{N}\sum_{i=1}^{N}(y_i-\hat y_i)^2
$$

In PyTorch:

```python
loss_fn = torch.nn.MSELoss()

loss = loss_fn(y_pred, y)
```

The loss tells us:

> **How far are the predictions from the actual targets?**

---

# 8. Backpropagation 🔥🔥🔥

After calculating the loss:

```python
loss.backward()
```

PyTorch's **Autograd** calculates the gradients.

Specifically:

$$
\frac{\partial L}{\partial w}
$$

and:

$$
\frac{\partial L}{\partial b}
$$

The gradients are stored in:

```python
w.grad
b.grad
```

### Flow

```text
Loss
 ↓
loss.backward()
 ↓
Autograd
 ↓
∂L/∂w
∂L/∂b
 ↓
w.grad
b.grad
```

---

# 9. Gradient Descent Update 🔥🔥🔥

The basic update rule is:

$$
w_{\text{new}}
=
w_{\text{old}}
-
\eta
\frac{\partial L}{\partial w}
$$

and:

$$
b_{\text{new}}
=
b_{\text{old}}
-
\eta
\frac{\partial L}{\partial b}
$$

where:

* $\eta$ = learning rate
* $\frac{\partial L}{\partial w}$ = weight gradient
* $\frac{\partial L}{\partial b}$ = bias gradient

The gradient tells us **which direction increases the loss**, so we move in the opposite direction.

---

# 10. Manual Parameter Update

For learning purposes, we can update manually:

```python
learning_rate = 0.01

with torch.no_grad():
    w -= learning_rate * w.grad
    b -= learning_rate * b.grad
```

### Why `torch.no_grad()`?

We don't want the parameter update operation itself to become part of the computational graph.

---

# 11. Gradient Accumulation ⚠️

PyTorch gradients **accumulate by default**.

Therefore, before calculating the next batch/iteration's gradients, we clear the old ones:

```python
w.grad.zero_()
b.grad.zero_()
```

The training cycle is:

```text
Forward
   ↓
Loss
   ↓
Clear old gradients
   ↓
Backward
   ↓
Gradients
   ↓
Update parameters
```

---

# 12. Using an Optimizer 🔥🔥🔥

Instead of manually implementing gradient descent, PyTorch provides optimizers.

```python
optimizer = torch.optim.SGD(
    [w, b],
    lr=0.01
)
```

Then:

```python
optimizer.zero_grad()

loss.backward()

optimizer.step()
```

### Meaning

```text
optimizer.zero_grad()
        ↓
Clear old gradients

loss.backward()
        ↓
Calculate gradients

optimizer.step()
        ↓
Update parameters
```

---

# 13. Complete Implementation ⭐⭐⭐

```python
import torch

# Data
X = torch.tensor([1., 2., 3., 4., 5.])
y = torch.tensor([3., 5., 7., 9., 11.])

# Learnable parameters
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

# Loss
loss_fn = torch.nn.MSELoss()

# Optimizer
optimizer = torch.optim.SGD(
    [w, b],
    lr=0.01
)

# Training
for epoch in range(1000):

    # Forward pass
    y_pred = w * X + b

    # Loss
    loss = loss_fn(y_pred, y)

    # Clear old gradients
    optimizer.zero_grad()

    # Backpropagation
    loss.backward()

    # Update parameters
    optimizer.step()

    if epoch % 100 == 0:
        print(
            f"Epoch {epoch}, "
            f"Loss: {loss.item():.4f}, "
            f"w: {w.item():.4f}, "
            f"b: {b.item():.4f}"
        )
```

After training:

$$
w\approx2
$$

and:

$$
b\approx1
$$

So the neuron has learned:

$$
\hat y\approx2x+1
$$

---

# 14. Inference

After training, we can give the neuron a new input:

```python
x_new = torch.tensor(10.)

prediction = w * x_new + b

print(prediction)
```

Expected:

$$
\hat y\approx21
$$

because:

$$
2(10)+1=21
$$

This is **inference**.

---

# 15. Training vs Inference

### Training

```text
Input
 ↓
Prediction
 ↓
Loss
 ↓
Backward
 ↓
Gradient
 ↓
Optimizer
 ↓
Update
```

### Inference

```text
Input
 ↓
Prediction
```

No parameter update is performed.

---

# 16. Manual Update vs Optimizer

| Manual implementation      | Optimizer                    |
| -------------------------- | ---------------------------- |
| `w -= lr * w.grad`         | `optimizer.step()`           |
| `w.grad.zero_()`           | `optimizer.zero_grad()`      |
| You implement update logic | PyTorch optimizer handles it |
| Good for understanding     | Used in practical projects   |

For real neural networks with thousands or millions of parameters, optimizers are much more convenient.

---

# 17. The Complete Learning Cycle 🔥

```text
             Training Data
                  │
                  ▼
             Input X
                  │
                  ▼
          Single Neuron
            ŷ = wx+b
                  │
                  ▼
             Prediction
                  │
                  ▼
                Loss
                  │
                  ▼
          loss.backward()
                  │
                  ▼
              Gradients
            /           \
        ∂L/∂w          ∂L/∂b
            \           /
             ▼         ▼
              Optimizer
                  │
                  ▼
              Update w,b
                  │
                  └──────────┐
                             │
                             ▼
                          Repeat
```

---

# 18. Learning Rate Experiment 🔬

Try different learning rates:

```python
lr = 0.001
```

```python
lr = 0.01
```

```python
lr = 0.1
```

The learning rate controls **how large each parameter update is**.

### Too small

```text
Very small updates
      ↓
Slow learning
```

### Reasonable

```text
Stable updates
      ↓
Good convergence
```

### Too large

```text
Huge updates
      ↓
Overshooting / instability
```

This connects directly to the optimization concepts from Module 12.

---

# 19. Why This Tiny Model Matters 🔥🔥🔥

This single neuron contains the fundamental mechanism behind much larger networks.

A large ANN might look like:

```text
Input
 ↓
Linear
 ↓
ReLU
 ↓
Linear
 ↓
ReLU
 ↓
Linear
 ↓
Output
```

But every `Linear` layer still performs:

$$
Z=XW+b
$$

And the entire network still follows:

```text
Forward
 ↓
Loss
 ↓
Backward
 ↓
Gradients
 ↓
Optimizer
 ↓
Parameter update
```

So the single neuron is the **foundation for everything we'll implement next**.

---

# 🎯 Must Know vs Useful

## 🔥🔥🔥 MUST KNOW

* Single neuron equation:

$$
\hat y=wx+b
$$

* Parameters: $w,b$
* `requires_grad=True`
* Forward pass
* Loss
* `loss.backward()`
* `.grad`
* `optimizer.zero_grad()`
* `optimizer.step()`
* Learning rate
* Parameter update
* Training vs inference

## 🔥🔥 IMPORTANT

* Manual gradient update
* Why `torch.no_grad()` is used for manual updates
* Gradient accumulation
* How changing learning rate affects convergence

## ➖ OPTIONAL FOR NOW

* Deriving the complete MSE gradients by hand
* Low-level Autograd internals

You already studied the calculus/backpropagation deeply in Module 6, so we don't need to repeat all of that here.

---

# 💻 Quick Reference

```python
# Parameters
w = torch.tensor(0., requires_grad=True)
b = torch.tensor(0., requires_grad=True)

# Optimizer
optimizer = torch.optim.SGD(
    [w, b],
    lr=0.01
)

# Training
for epoch in range(1000):

    # Forward
    y_pred = w * X + b

    # Loss
    loss = loss_fn(y_pred, y)

    # Backward
    optimizer.zero_grad()
    loss.backward()

    # Update
    optimizer.step()
```

### 🧠 Remember:

> **Forward computes the prediction → loss measures the error → backward computes gradients → optimizer updates the parameters.**

---

